In [3]:
import torch

%run DecisionTreeClassifier.ipynb

X shape: torch.Size([300, 2])
y shape: torch.Size([300])
Classes: tensor([0, 1, 2])


# Random Forest

## Definition

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to improve accuracy and reduce overfitting. Each tree is trained on a different bootstrap sample of the data and considers a random subset of features at each split. The final prediction is obtained by majority voting (classification) or averaging (regression).

## Algorithm Steps

1. Choose the number of trees (`n_trees`).
2. For each tree:
   - Generate a bootstrap sample from the training data.
   - Train a Decision Tree using a random subset of features at each split.
   - Store the trained tree.
3. During prediction:
   - Collect predictions from all trees.
   - Return the majority vote (classification) or average (regression).

# Bootstrap Sampling

### **Bootstrap Sampling** is a resampling technique that creates a new dataset by randomly selecting samples **with replacement** from the original dataset, allowing some samples to appear multiple times while others may not appear at all.

In [7]:
class BootstrapSampling:
    def __call__(self, X, y):
        n_samples = X.shape[0]

        indices = torch.randint(
            low=0,
            high=n_samples,
            size=(n_samples,)
        )

        return X[indices], y[indices]

In [41]:

class RandomForest:
    def __init__(self, n_trees=100,max_depth=5):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.bootstrap = BootstrapSampling()
        self.trees = []

    def fit(self, X, y):
        for _ in range(self.n_trees):

            X_sample, y_sample = self.bootstrap(X, y)

            tree = DecisionTree(max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)

            self.trees.append(tree)

    def predict(self,x):
        prediction = []
        for tree in self.trees:
            pred = tree.predict(x)
            prediction.append(pred)

        prediction = torch.stack(prediction) # Converting normal list into tensor

        return torch.mode(prediction,dim=0).values # .values only return prediction and not the indices


In [42]:
model = RandomForest()
model.fit(X,y)

In [ ]:
## Testing Data as per the orignal data, which our RandomForest object used, from DecisionTreeClassifie.ipynb

X_test = torch.tensor([
    [-3.2, -2.8],   # Class 0
    [-2.5, -3.6],   # Class 0
    [-4.1, -2.9],   # Class 0

    [ 3.1, -2.7],   # Class 1
    [ 2.6, -3.8],   # Class 1
    [ 4.0, -2.5],   # Class 1

    [ 0.2,  3.4],   # Class 2
    [-0.8,  2.7],   # Class 2
    [ 1.0,  4.2],   # Class 2

    [ 0.0,  3.0],   # Class 2
], dtype=torch.float32)

y_test = torch.tensor([
    0, 0, 0,
    1, 1, 1,
    2, 2, 2, 2
], dtype=torch.long)

In [43]:
y_pred = model.predict(X_test)
print("Predictions :", y_pred)
print("Ground Truth:", y_test)

Predictions : tensor([0, 0, 0, 1, 1, 1, 2, 2, 2, 2])
Ground Truth: tensor([0, 0, 0, 1, 1, 1, 2, 2, 2, 2])
